# 7.2 Tensors, Shapes, and Core PyTorch Workflow

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook moves from high-level framing into the day-to-day PyTorch objects you manipulate in real experiments: tensors, batches, dataloaders, device placement, and forward passes.


## 7.2.1 Tensors versus NumPy arrays

### Why It Matters
PyTorch tensors feel similar to NumPy arrays, but they also carry autograd and device semantics. Start by checking the same numbers represented in both structures.


In [ ]:
import numpy as np
import torch

arr = np.array([[1.0, 2.0], [3.0, 4.0]])
ten = torch.tensor(arr)

{
    "numpy_mean": float(arr.mean()),
    "tensor_mean": float(ten.mean().item()),
    "tensor_dtype": str(ten.dtype),
}


### Interpretation
The data container looks familiar, which lowers the entry cost. The extra value comes later when gradients and devices enter the workflow.


## 7.2.2 Shapes, batching, and broadcasting

### Why It Matters
Most neural-network bugs are shape bugs. Broadcasting can be powerful, but you need to know which dimension is the batch and which dimensions represent features or channels.


In [ ]:
import torch

batch = torch.arange(12, dtype=torch.float32).reshape(3, 4)
bias = torch.tensor([0.5, -0.5, 1.0, 2.0])
shifted = batch + bias

{
    "batch_shape": list(batch.shape),
    "bias_shape": list(bias.shape),
    "shifted_first_row": shifted[0].tolist(),
}


### Interpretation
The bias vector is broadcast across rows, which is the same idea used by linear layers internally.


## 7.2.3 Datasets, dataloaders, and mini-batches

### Why It Matters
Training rarely happens on the entire dataset at once. A dataloader packages examples into shuffled mini-batches and becomes the standard input loop for training code.


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(20, 5)
y = (X.sum(dim=1) > 0).long()
loader = DataLoader(TensorDataset(X, y), batch_size=6, shuffle=True)
batch_X, batch_y = next(iter(loader))

{
    "num_batches": len(loader),
    "batch_X_shape": list(batch_X.shape),
    "batch_y_shape": list(batch_y.shape),
}


### Interpretation
Mini-batches control memory use and create the stochastic gradient updates used in modern neural-network training.


## 7.2.4 CPU versus device movement

### Why It Matters
Tensors and models must live on the same device. Even if you stay on CPU for this course, it helps to make device movement explicit in your code.


In [ ]:
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X = torch.randn(4, 3).to(device)
model = nn.Linear(3, 2).to(device)
out = model(X)

{
    "selected_device": str(device),
    "input_device": str(X.device),
    "output_shape": list(out.shape),
}


### Interpretation
Writing device-aware code early avoids a common class of runtime errors later when models become larger.


## 7.2.5 Forward pass intuition

### Why It Matters
A forward pass means taking inputs and pushing them through the current network weights to get outputs. This cell shows that process with a tiny hand-built module.


In [ ]:
import torch
from torch import nn

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(3, 4)
        self.output = nn.Linear(4, 1)

    def forward(self, x):
        x = torch.relu(self.hidden(x))
        return self.output(x)

net = TinyNet()
x = torch.tensor([[0.2, -1.0, 0.5], [1.2, 0.3, -0.7]])
preds = net(x)
{
    "pred_shape": list(preds.shape),
    "pred_values": preds.detach().flatten().round(decimals=3).tolist(),
}


### Interpretation
The forward method is the executable definition of the model’s computation graph.


## 7.2.6 A minimal training-ready workflow

### Why It Matters
A complete PyTorch workflow combines tensors, a dataloader, a model, a loss, and an optimizer. This cell condenses that loop into one short example.


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(12)
X = torch.randn(80, 4)
y = ((X[:, 0] - 0.8 * X[:, 1] + 0.5 * X[:, 2]) > 0).float().unsqueeze(1)
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for batch_X, batch_y in loader:
    optimizer.zero_grad()
    loss = loss_fn(model(batch_X), batch_y)
    loss.backward()
    optimizer.step()

{
    "last_batch_loss": round(float(loss.item()), 4),
    "num_parameters": sum(p.numel() for p in model.parameters()),
}


### Interpretation
This pattern repeats throughout the rest of the course, with larger models and richer data.
